In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    T5Tokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, pipeline
)
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
# Load fine-tuned PubMedBERT encoder
print("Loading PubMedBERT encoder...")
encoder_path = "./saved_pubmedbert_pico_model/"
encoder_tokenizer = AutoTokenizer.from_pretrained(encoder_path)
encoder_model = AutoModelForSequenceClassification.from_pretrained(encoder_path).to(device)
encoder_model.eval()

# Load BioMistral decoder
print("Loading BioMistral-7B decoder...")
biomistral_name = "BioMistral/BioMistral-7B"
biomistral_tokenizer = AutoTokenizer.from_pretrained(biomistral_name)
biomistral_model = AutoModelForCausalLM.from_pretrained(
    biomistral_name,
    torch_dtype=torch.float16,
    device_map="auto",
    use_safetensors=False
)
generator = pipeline("text-generation", model=biomistral_model, tokenizer=biomistral_tokenizer)

LABEL_MAP = {0: "Complete", 1: "Missing P", 2: "Missing I", 3: "Missing O"}
ELEMENT_MAP = {"P": "Patient/Population", "I": "Intervention", "O": "Outcome"}


def predict_missing_pico(text):
    inputs = encoder_tokenizer(text, max_length=512, padding='max_length', truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = encoder_model(**inputs)
    return LABEL_MAP[torch.argmax(outputs.logits, dim=1).item()]


def build_pico_prompt(abstract_text, prediction):
    system = "You are an expert clinical data extractor and reviewer. Your task is to analyze a medical abstract based on findings from an upstream classification model.\n\n"
    if prediction == "Complete":
        task = "The classification model found all required PICO elements. Output a short, structured summary of Patient, Intervention, and Outcome based strictly on the text. Do not hallucinate."
    else:
        missing = prediction.split(" ")[1]
        full_name = ELEMENT_MAP.get(missing, "PICO element")
        task = f"The abstract is MISSING the {full_name} element. Generate a single, professional clarification question requesting the missing {full_name} details. Do not answer it yourself."
    return f"{system}--- TASK ---\n{task}\n\n--- ABSTRACT ---\n{abstract_text}\n\n--- RESPONSE ---\n"


# Load training data
print("\nLoading masked training data...")
with open("ebm_nlp_train_mixed.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(f"Total training examples: {len(train_data)}")

# Generate silver labels — this takes ~2-4 hours on a single GPU for ~4800 examples
silver_labels = []
print("\nGenerating silver labels with BioMistral...")
for i, item in enumerate(train_data):
    abstract = item["model_input_text"]
    prediction = predict_missing_pico(abstract)
    prompt = build_pico_prompt(abstract, prediction)

    output = generator(
        prompt,
        max_new_tokens=75,
        do_sample=True,
        temperature=0.3,
        return_full_text=False
    )
    target = output[0]['generated_text'].strip()

    silver_labels.append({
        "pmid": item["pmid"],
        "abstract": abstract,
        "prediction": prediction,
        "target_question": target,
        "is_masked": item["is_masked"],
        "missing_slot": item["missing_slot"]
    })

    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(train_data)}")
        # Save checkpoint every 100
        with open("silver_labels_checkpoint.json", "w") as f:
            json.dump(silver_labels, f, indent=2)

# Final save
with open("silver_labels_train.json", "w") as f:
    json.dump(silver_labels, f, indent=2)

print(f"\nSaved {len(silver_labels)} silver labels to silver_labels_train.json")

# Free memory before training
del biomistral_model, encoder_model, generator
torch.cuda.empty_cache()

In [ ]:
# Reload models for test silver labels
encoder_model = AutoModelForSequenceClassification.from_pretrained(encoder_path).to(device)
encoder_model.eval()
biomistral_model = AutoModelForCausalLM.from_pretrained(
    biomistral_name, torch_dtype=torch.float16, device_map="auto", use_safetensors=False
)
generator = pipeline("text-generation", model=biomistral_model, tokenizer=biomistral_tokenizer)

with open("ebm_nlp_test_mixed.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

silver_labels_test = []
print(f"Generating silver labels for {len(test_data)} test examples...")
for i, item in enumerate(test_data):
    abstract = item["model_input_text"]
    prediction = predict_missing_pico(abstract)
    prompt = build_pico_prompt(abstract, prediction)
    output = generator(prompt, max_new_tokens=75, do_sample=True, temperature=0.3, return_full_text=False)
    silver_labels_test.append({
        "pmid": item["pmid"],
        "abstract": abstract,
        "prediction": prediction,
        "target_question": output[0]['generated_text'].strip(),
        "is_masked": item["is_masked"],
        "missing_slot": item["missing_slot"]
    })
    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(test_data)}")

with open("silver_labels_test.json", "w") as f:
    json.dump(silver_labels_test, f, indent=2)
print(f"Saved {len(silver_labels_test)} test silver labels.")

del biomistral_model, encoder_model, generator
torch.cuda.empty_cache()

In [ ]:


TASK_PREFIX_MISSING = "You are an expert clinical reviewer. The following medical abstract is missing key clinical information. Ask a single, targeted clarification question about what is missing.\n\nAbstract: "
TASK_PREFIX_COMPLETE = "You are an expert clinical reviewer. Extract a structured PICO summary (Patient, Intervention, Outcome) from the following medical abstract.\n\nAbstract: "

with open("silver_labels_train.json") as f:
    train_silver = json.load(f)
with open("silver_labels_test.json") as f:
    test_silver = json.load(f)

# Hold out 5% of train as validation
np.random.seed(42)
indices = np.random.permutation(len(train_silver))
val_size = len(train_silver) // 20
val_indices = set(indices[:val_size].tolist())

train_examples = []
val_examples = []

for i, item in enumerate(train_silver):
    prefix = TASK_PREFIX_COMPLETE if item["prediction"] == "Complete" else TASK_PREFIX_MISSING
    example = {
        "input_text": prefix + item["abstract"],
        "target_text": item["target_question"]
    }
    if i in val_indices:
        val_examples.append(example)
    else:
        train_examples.append(example)

test_examples = []
for item in test_silver:
    prefix = TASK_PREFIX_COMPLETE if item["prediction"] == "Complete" else TASK_PREFIX_MISSING
    test_examples.append({
        "input_text": prefix + item["abstract"],
        "target_text": item["target_question"]
    })

print(f"Train: {len(train_examples)} | Val: {len(val_examples)} | Test: {len(test_examples)}")

In [ ]:
MODEL_NAME = "google/flan-t5-base"
MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 96

print(f"Loading FLAN-T5 tokenizer: {MODEL_NAME}")
flan_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class PICOSeq2SeqDataset(Dataset):
    def __init__(self, examples, tokenizer, max_input_len=512, max_target_len=96):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        model_inputs = self.tokenizer(
            ex["input_text"],
            max_length=self.max_input_len,
            truncation=True,
            padding=False
        )
        labels = self.tokenizer(
            ex["target_text"],
            max_length=self.max_target_len,
            truncation=True,
            padding=False
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs


train_ds = PICOSeq2SeqDataset(train_examples, flan_tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
val_ds = PICOSeq2SeqDataset(val_examples, flan_tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
test_ds = PICOSeq2SeqDataset(test_examples, flan_tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
print(f"Datasets ready. Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
print(f"Loading FLAN-T5 base model...")
flan_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

OUTPUT_DIR = "./saved_flan_t5_pico_model/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Match Mihir's hyperparameters
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    warmup_ratio=0.05,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=50,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=flan_tokenizer,
    model=flan_model,
    padding="longest",
    label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=flan_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=flan_tokenizer,
    data_collator=data_collator
)

print("Starting fine-tuning...")
trainer.train()

print("Saving final model...")
trainer.save_model(OUTPUT_DIR)
flan_tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
flan_model.eval()
test_predictions = []

print("Generating predictions on test set...")
for i, ex in enumerate(test_examples):
    inputs = flan_tokenizer(
        ex["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = flan_model.generate(
            **inputs,
            num_beams=4,
            max_new_tokens=96,
            length_penalty=1.0,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    pred = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    test_predictions.append({
        "input": ex["input_text"],
        "reference": ex["target_text"],
        "generated": pred
    })

    if (i + 1) % 25 == 0:
        print(f"  Generated {i+1}/{len(test_examples)}")

df_results = pd.DataFrame(test_predictions)
df_results.to_csv("flan_t5_pico_test_results.csv", index=False)
print(f"Saved {len(df_results)} predictions to flan_t5_pico_test_results.csv")
df_results.head(3)

In [ ]:
# Install if needed: pip install bert-score sacrebleu rouge-score
from bert_score import score as bertscore_fn
import sacrebleu
from rouge_score import rouge_scorer

references = [r["reference"] for r in test_predictions]
generated = [r["generated"] for r in test_predictions]

print("Computing BERTScore...")
P, R, F1 = bertscore_fn(generated, references, model_type="microsoft/deberta-xlarge-mnli", lang="en", verbose=False)
bertscore_p, bertscore_r, bertscore_f1 = P.mean().item(), R.mean().item(), F1.mean().item()

print("Computing ROUGE...")
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
for ref, gen in zip(references, generated):
    scores = scorer.score(ref, gen)
    for k in rouge_scores:
        rouge_scores[k].append(scores[k].fmeasure)
rouge_avg = {k: np.mean(v) for k, v in rouge_scores.items()}

print("Computing BLEU...")
bleu = sacrebleu.corpus_bleu(generated, [references]).score

# Question vs Statement analysis (Mihir's metric)
gen_questions = sum(1 for g in generated if "?" in g)
ref_questions = sum(1 for r in references if "?" in r)

f1_per_example = F1.tolist()
gen_q_f1 = np.mean([f1_per_example[i] for i, g in enumerate(generated) if "?" in g]) if gen_questions > 0 else 0
gen_s_f1 = np.mean([f1_per_example[i] for i, g in enumerate(generated) if "?" not in g]) if (len(generated) - gen_questions) > 0 else 0
ref_q_f1 = np.mean([f1_per_example[i] for i, r in enumerate(references) if "?" in r]) if ref_questions > 0 else 0
ref_s_f1 = np.mean([f1_per_example[i] for i, r in enumerate(references) if "?" not in r]) if (len(references) - ref_questions) > 0 else 0

results = {
    "test_size": len(test_predictions),
    "bertscore": {"precision": bertscore_p, "recall": bertscore_r, "f1": bertscore_f1},
    "rouge": rouge_avg,
    "bleu": bleu,
    "question_analysis": {
        "generated_questions_pct": gen_questions / len(generated) * 100,
        "reference_questions_pct": ref_questions / len(references) * 100,
        "generated_questions_count": gen_questions,
        "reference_questions_count": ref_questions,
        "bertscore_f1_when_generated_is_question": gen_q_f1,
        "bertscore_f1_when_generated_is_statement": gen_s_f1,
        "bertscore_f1_when_reference_is_question": ref_q_f1,
        "bertscore_f1_when_reference_is_statement": ref_s_f1,
    }
}

with open("flan_t5_pico_evaluation.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "="*60)
print("FLAN-T5 PICO FINE-TUNING — EVALUATION RESULTS")
print("="*60)
print(f"BERTScore F1:    {bertscore_f1:.4f}  (Mihir's MedDialog: 0.5970)")
print(f"BERTScore P / R: {bertscore_p:.4f} / {bertscore_r:.4f}")
print(f"ROUGE-1 / 2 / L: {rouge_avg['rouge1']:.4f} / {rouge_avg['rouge2']:.4f} / {rouge_avg['rougeL']:.4f}")
print(f"BLEU:            {bleu:.2f}")
print(f"\nQUESTION ANALYSIS (the key metric):")
print(f"  Generated questions: {results['question_analysis']['generated_questions_pct']:.2f}% ({gen_questions}/{len(generated)})")
print(f"  Reference questions: {results['question_analysis']['reference_questions_pct']:.2f}% ({ref_questions}/{len(references)})")
print(f"  Mihir's MedDialog comparison: 0.04% generated vs 6.15% reference")
print("="*60)